# 12 — RLHF: Reward Modeling

RLHF (Reinforcement Learning from Human Feedback) is how modern LLMs are aligned to human preferences. The standard pipeline has two stages: (1) train a reward model from human preference data, (2) fine-tune the LLM with PPO against that reward model. This notebook covers stage 1.

## Why RLHF

Supervised fine-tuning teaches an LLM to imitate demonstrations — but demonstrations cannot fully specify what we want. It is easier for humans to *compare* two outputs than to write the ideal output from scratch.

RLHF leverages pairwise comparisons:
1. Collect paired comparisons: for the same prompt, humans label which of two responses they prefer
2. Train a **reward model** to predict human preference scores
3. Fine-tune the LLM to generate responses that score highly on the reward model

This was the key innovation in InstructGPT (OpenAI, 2022), and the same pipeline underlies Claude, GPT-4, Gemini, and Llama 2/3.

## The Bradley-Terry model

Human preferences are modelled as:
$$P(A \succ B) = \sigma(r(x, y_A) - r(x, y_B))$$

The reward model is trained with cross-entropy loss:
$$\mathcal{L}_{RM} = -\mathbb{E}\left[\log \sigma(r(x, y_w) - r(x, y_l))\right]$$

Where y_w is the preferred ('winner') response and y_l is the 'loser'.

**Failure modes to know**:
- **Overoptimisation**: PPO pushes the policy to find RM blind spots — outputs that score high but aren't actually preferred
- **Sycophancy**: RMs trained on human feedback can learn to prefer flattering, agreeable responses even when wrong
- **Distribution shift**: preferences from labellers during training may not match users at deployment

All motivate the KL penalty in RLHF stage 2 (NB 13).

In [ ]:
# Reward model training from synthetic preference pairs

import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42); np.random.seed(42)

def make_preference_data(n=5000, dim=128):
    w_true = torch.randn(dim); w_true /= w_true.norm()
    data = []
    for _ in range(n):
        prompt = torch.randn(dim)
        resp_a, resp_b = torch.randn(dim), torch.randn(dim)
        score_a = (prompt*resp_a).sum() + w_true@resp_a
        score_b = (prompt*resp_b).sum() + w_true@resp_b
        # 80% accurate human labels (20% noise)
        if torch.rand(1).item() < 0.8:
            chosen = resp_a if score_a > score_b else resp_b
            rejected = resp_b if score_a > score_b else resp_a
        else:
            chosen = resp_b if score_a > score_b else resp_a
            rejected = resp_a if score_a > score_b else resp_b
        data.append((torch.cat([prompt, chosen]), torch.cat([prompt, rejected])))
    return data, w_true

class PreferenceDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

class RewardModel(nn.Module):
    def __init__(self, input_dim=256, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

def bradley_terry_loss(r_chosen, r_rejected):
    return -torch.log(torch.sigmoid(r_chosen - r_rejected) + 1e-8).mean()

data, w_true = make_preference_data(n=5000, dim=128)
n_train = int(0.8 * len(data))
train_loader = DataLoader(PreferenceDataset(data[:n_train]), batch_size=64, shuffle=True)
val_loader = DataLoader(PreferenceDataset(data[n_train:]), batch_size=256)

rm = RewardModel(input_dim=256)
optimizer = optim.AdamW(rm.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

train_losses, val_losses, val_accs = [], [], []
for epoch in range(30):
    rm.train()
    ep_losses = []
    for chosen, rejected in train_loader:
        loss = bradley_terry_loss(rm(chosen), rm(rejected))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep_losses.append(loss.item())
    scheduler.step(); train_losses.append(np.mean(ep_losses))

    rm.eval(); val_loss = 0; correct = total = 0
    with torch.no_grad():
        for chosen, rejected in val_loader:
            rc, rr = rm(chosen), rm(rejected)
            val_loss += bradley_terry_loss(rc, rr).item()
            correct += (rc > rr).sum().item(); total += len(rc)
    val_losses.append(val_loss / len(val_loader)); val_accs.append(correct / total)
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:3d} | Train: {train_losses[-1]:.4f} | Val: {val_losses[-1]:.4f} | Acc: {val_accs[-1]:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train', color='steelblue')
ax1.plot(val_losses, label='Val', color='darkorange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BT Loss')
ax1.set_title('Reward Model Training', fontweight='bold'); ax1.legend()
ax2.plot(val_accs, color='green')
ax2.axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Label noise ceiling (80%)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Preference Accuracy')
ax2.set_title('Reward Model Val Accuracy', fontweight='bold'); ax2.legend()
plt.tight_layout(); plt.savefig('/tmp/reward_model.png', dpi=100, bbox_inches='tight'); plt.show()
print(f"Final val accuracy: {val_accs[-1]:.3f}")
